In [1]:
# Packages
import duckdb
import os
import pandas as pd
# Local functions
from mimic_pipeline.duck import *
from mimic_pipeline.icustays import *
from mimic_pipeline.io_registry import register_parquet, register_parquet_general

## Exploratory queries to guide feature selection

In [2]:
# Prep ddb connection for data exploration
ddb = duckdb.connect("data/data.duckdb")
ddb.execute(f"PRAGMA memory_limit='{"8GB"}';")
ddb.execute(f"PRAGMA temp_directory='data/.duckdb_tmp';")

In [3]:
# Load base parquet files
base_path = "data/base_parquet/"
directory_map = {
    # core
    "ADMISSIONS": "admissions.parquet",
    "PATIENTS": "patients.parquet",

    # hospital
    "DIAGNOSES_ICD": "diagnoses_icd.parquet",
    "D_ICD_DIAGNOSES": "d_icd_diagnoses.parquet",

    # icu
    "ICUSTAYS": "icustays.parquet",
    "INPUTEVENTS": "inputevents.parquet",
    "OUTPUTEVENTS": "outputevents.parquet",
    "CHARTEVENTS": "chartevents.parquet",
    "D_ITEMS": "d_items.parquet",
}
filepath_map = {
    key: base_path + value for key, value in directory_map.items()
}
register_parquet(ddb, filepath_map)

Done: Loaded ADMISSIONS table in 1.24s
Done: Loaded PATIENTS table in 0.97s
Done: Loaded DIAGNOSES_ICD table in 0.84s
Done: Loaded D_ICD_DIAGNOSES table in 0.06s
Done: Loaded ICUSTAYS table in 0.05s
Done: Loaded INPUTEVENTS table in 3.26s
Done: Loaded OUTPUTEVENTS table in 1.76s
Done: Loaded CHARTEVENTS table in 13.63s
Done: Loaded D_ITEMS table in 0.02s


In [4]:
# Load preprocessing parquet files
base_path = "data/preprocessing_checkpoints/"
directory_map = {
    "BAD_ADMISSIONS": "bad_admissions.parquet",
    "BASE": "base.parquet",
    "CLEAN_CHARTEVENTS": "clean_chartevents.parquet",
    "CLEAN_INPUTEVENTS": "clean_inputevents.parquet",
    "CLEAN_OUTPUTEVENTS": "clean_outputevents.parquet",
    "COHORT": "cohort.parquet",
    "DD_NORM": "dd_norm.parquet",
    "EXCLUDE_TITLES": "exclude_titles.parquet",
}
filepath_map = {
    key: base_path + value for key, value in directory_map.items()
}
register_parquet_general(ddb, filepath_map)

Done: Loaded BAD_ADMISSIONS table in 0.01s
Done: Loaded BASE table in 1.13s
Done: Loaded CLEAN_CHARTEVENTS table in 13.14s
Done: Loaded CLEAN_INPUTEVENTS table in 3.94s
Done: Loaded CLEAN_OUTPUTEVENTS table in 1.35s
Done: Loaded COHORT table in 0.86s
Done: Loaded DD_NORM table in 0.07s
Done: Loaded EXCLUDE_TITLES table in 0.01s


In [5]:
peek(ddb, "D_ITEMS", 5)

┌────────┬─────────────────────────┬────────────────────┬────────────────┬─────────────────────┬──────────┬───────────────┬────────────────┬─────────────────┐
│ itemid │          label          │    abbreviation    │    linksto     │      category       │ unitname │  param_type   │ lownormalvalue │ highnormalvalue │
│ int64  │         varchar         │      varchar       │    varchar     │       varchar       │ varchar  │    varchar    │     int64      │     double      │
├────────┼─────────────────────────┼────────────────────┼────────────────┼─────────────────────┼──────────┼───────────────┼────────────────┼─────────────────┤
│ 220003 │ ICU Admission date      │ ICU Admission date │ datetimeevents │ ADT                 │ NULL     │ Date and time │           NULL │            NULL │
│ 220045 │ Heart Rate              │ HR                 │ chartevents    │ Routine Vital Signs │ bpm      │ Numeric       │           NULL │            NULL │
│ 220046 │ Heart rate Alarm - High │ HR Alarm 

# Top Numeric Features in CHARTEVENTS

In [6]:
query = """
SELECT
    d.itemid,
    d.label,
    d.category,
    d.param_type,
    d.unitname,
    COUNT(c.value) AS event_count
FROM CLEAN_CHARTEVENTS c
JOIN D_ITEMS d ON c.itemid = d.itemid
WHERE
    d.param_type IN ('Numeric', 'Numeric/Calculated')
GROUP BY
    d.itemid, d.label, d.category, d.param_type, d.unitname
ORDER BY
    event_count DESC
LIMIT 200;
"""
ddb.query(query).show(max_rows=200)

┌────────┬──────────────────────────────────────────┬─────────────────────────────┬────────────┬─────────────────┬─────────────┐
│ itemid │                  label                   │          category           │ param_type │    unitname     │ event_count │
│ int64  │                 varchar                  │           varchar           │  varchar   │     varchar     │    int64    │
├────────┼──────────────────────────────────────────┼─────────────────────────────┼────────────┼─────────────────┼─────────────┤
│ 220045 │ Heart Rate                               │ Routine Vital Signs         │ Numeric    │ bpm             │     6773430 │
│ 220210 │ Respiratory Rate                         │ Respiratory                 │ Numeric    │ insp/min        │     6717097 │
│ 220277 │ O2 saturation pulseoxymetry              │ Respiratory                 │ Numeric    │ %               │     6638357 │
│ 220181 │ Non Invasive Blood Pressure mean         │ Routine Vital Signs         │ Numeric    │ 

# Top Features in INPUTEVENTS

In [7]:
query = """
SELECT
    d.itemid,
    d.label,
    d.category,
    d.unitname,
    COUNT(*) AS event_count
FROM CLEAN_INPUTEVENTS i
JOIN D_ITEMS d ON i.itemid = d.itemid
GROUP BY
    d.itemid, d.label, d.category, d.unitname
ORDER BY
    event_count DESC
LIMIT 100;
"""
ddb.query(query).show(max_rows=100)

┌────────┬─────────────────────────────────┬─────────────────────────┬──────────┬─────────────┐
│ itemid │              label              │        category         │ unitname │ event_count │
│ int64  │             varchar             │         varchar         │ varchar  │    int64    │
├────────┼─────────────────────────────────┼─────────────────────────┼──────────┼─────────────┤
│ 225158 │ NaCl 0.9%                       │ Fluids/Intake           │ mL       │     1326336 │
│ 220949 │ Dextrose 5%                     │ Fluids/Intake           │ mL       │     1087609 │
│ 225943 │ Solution                        │ Fluids/Intake           │ mL       │      593136 │
│ 226452 │ PO Intake                       │ Fluids/Intake           │ mL       │      446252 │
│ 222168 │ Propofol                        │ Medications             │ mg       │      420185 │
│ 221906 │ Norepinephrine                  │ Medications             │ mg       │      353659 │
│ 225799 │ Gastric Meds                 

# Top Features in OUTPUTEVENTS

In [8]:
query = """
SELECT
    d.itemid,
    d.label,
    d.category,
    d.unitname,
    COUNT(*) AS event_count
FROM CLEAN_OUTPUTEVENTS o
JOIN D_ITEMS d ON o.itemid = d.itemid
GROUP BY
    d.itemid, d.label, d.category, d.unitname
ORDER BY
    event_count DESC
LIMIT 100;
"""
ddb.query(query).show(max_rows=100)

┌────────┬──────────────────────────────┬──────────┬──────────┬─────────────┐
│ itemid │            label             │ category │ unitname │ event_count │
│ int64  │           varchar            │ varchar  │ varchar  │    int64    │
├────────┼──────────────────────────────┼──────────┼──────────┼─────────────┤
│ 226559 │ Foley                        │ Output   │ mL       │     3137879 │
│ 226560 │ Void                         │ Output   │ mL       │      280250 │
│ 226588 │ Chest Tube #1                │ Output   │ mL       │      274307 │
│ 226606 │ Cerebral Ventricular #1      │ Drains   │ mL       │      103020 │
│ 227510 │ TF Residual                  │ Output   │ mL       │       91173 │
│ 226599 │ Jackson Pratt #1             │ Drains   │ mL       │       63052 │
│ 226561 │ Condom Cath                  │ Output   │ mL       │       41896 │
│ 226589 │ Chest Tube #2                │ Output   │ mL       │       40864 │
│ 226575 │ Nasogastric                  │ Output   │ mL       │ 

# Values For a Specific itemid

In [9]:
peek(ddb, "CLEAN_CHARTEVENTS", 1)

┌────────────┬──────────┬──────────┬─────────────────────┬────────┬─────────┬──────────┬──────────┐
│ subject_id │ hadm_id  │ stay_id  │      charttime      │ itemid │  value  │ valuenum │ valueuom │
│   int64    │  int64   │  int64   │      timestamp      │ int64  │ varchar │  double  │ varchar  │
├────────────┼──────────┼──────────┼─────────────────────┼────────┼─────────┼──────────┼──────────┤
│   10003700 │ 28623837 │ 30600691 │ 2165-04-24 05:10:00 │ 228236 │ 0       │      0.0 │ NULL     │
└────────────┴──────────┴──────────┴─────────────────────┴────────┴─────────┴──────────┴──────────┘



In [10]:
# Using heart rate for this example
query = """
SELECT
    value,
    COUNT(value) as value_count
FROM CLEAN_CHARTEVENTS
WHERE
    itemid = 220045
GROUP BY
    value
ORDER BY
    value_count DESC
LIMIT 20;
"""
ddb.query(query).show()

┌─────────┬─────────────┐
│  value  │ value_count │
│ varchar │    int64    │
├─────────┼─────────────┤
│ 80      │      196453 │
│ 88      │      151621 │
│ 82      │      149628 │
│ 85      │      149208 │
│ 81      │      146673 │
│ 84      │      146663 │
│ 70      │      145771 │
│ 87      │      144369 │
│ 86      │      143264 │
│ 83      │      142798 │
│ 90      │      141867 │
│ 79      │      141330 │
│ 78      │      140894 │
│ 77      │      137782 │
│ 76      │      137774 │
│ 75      │      137594 │
│ 89      │      136257 │
│ 74      │      130369 │
│ 91      │      129527 │
│ 73      │      127972 │
├─────────┴─────────────┤
│ 20 rows     2 columns │
└───────────────────────┘



# Test Timeseries Creation

In [11]:
from mimic_pipeline.feature_selection import chartevents_avg_features, inputevents_sum_features, outputevents_sum_features

Creates rows per 4 hour bin during patient stay for each patient using ICUSTAYS

In [12]:
# Create the base table of all patient time bins from ICUSTAYS
from mimic_pipeline.feature_selection import create_patient_bins
create_patient_bins(ddb, from_table="ICUSTAYS", output_table="ALL_PATIENT_BINS")
peek(ddb, "ALL_PATIENT_BINS", 10)

Created ALL_PATIENT_BINS.
   └─ wrote all_patient_bins.parquet
┌──────────┬─────────────────────┬─────────────────────┬────────────────┬─────────────────────┬─────────────────────┐
│ stay_id  │       intime        │       outtime       │ time_bin_index │   bin_start_time    │    bin_end_time     │
│  int64   │      timestamp      │      timestamp      │     int64      │      timestamp      │      timestamp      │
├──────────┼─────────────────────┼─────────────────────┼────────────────┼─────────────────────┼─────────────────────┤
│ 31793211 │ 2154-03-03 04:11:00 │ 2154-03-04 18:16:56 │              0 │ 2154-03-03 04:11:00 │ 2154-03-03 08:11:00 │
│ 31793211 │ 2154-03-03 04:11:00 │ 2154-03-04 18:16:56 │              1 │ 2154-03-03 08:11:00 │ 2154-03-03 12:11:00 │
│ 31793211 │ 2154-03-03 04:11:00 │ 2154-03-04 18:16:56 │              2 │ 2154-03-03 12:11:00 │ 2154-03-03 16:11:00 │
│ 31793211 │ 2154-03-03 04:11:00 │ 2154-03-04 18:16:56 │              3 │ 2154-03-03 16:11:00 │ 2154-03-03 20:1

Aggregate and pivot dynamic features from CHARTEVENTS

In [13]:
from mimic_pipeline.feature_selection import downselect_chartevents
downselect_chartevents(ddb, from_table="CLEAN_CHARTEVENTS", output_table="CHARTEVENTS_PIVOTED")

Created CHARTEVENTS_PIVOTED.
   └─ wrote chartevents_pivoted.parquet


In [14]:
peek(ddb, "CHARTEVENTS_PIVOTED")

┌──────────┬────────────────┬───────────┬──────────┬────────┬────────┬───────────┬───────────────┬────────────────┬───────────────────┬────────────────────┬───────────────┬────────┬────────┬───────────────┬────────┬────────┬──────────────────────┬───────────────────┬─────────────────┬────────────────┬────────────────┬──────────────┬────────────┬──────────┬────────────────────┬─────────────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────────┬───────────────┬────────────────────┬───────────┬──────────┬──────────────┬────────────────┬───────────────┬────────────┬────────┬───────────┬────────┬────────────┬───────────────────────┬─────────────────┬────────┬─────────────┬────────────────┬───────────────┬───────────────┬───────────┬─────────┬────────────────────┬───────────┬────────┬───────────────────┬───────────────────┬──────────────┬───────────────┬──────────────┬──────────┬────────────────────┬───────────────────┬──────────────────────┬────────────────

Fill the NULL entries

In [15]:
from mimic_pipeline.feature_selection import build_zero_fill_select_body, zero_fill

zero_fill(ddb,
          select_body=build_zero_fill_select_body(chartevents_avg_features.values()),
          from_table="CHARTEVENTS_PIVOTED",
          output_table="CHARTEVENTS_ZEROED",
          print_query=True)


Query:

    CREATE OR REPLACE TABLE CHARTEVENTS_ZEROED AS
    SELECT
        stay_id,
        time_bin_index,
        COALESCE(HeartRate, 0.0) AS HeartRate_value,
CASE WHEN HeartRate IS NULL THEN 1 ELSE 0 END AS HeartRate_missing,
COALESCE(RespRate, 0.0) AS RespRate_value,
CASE WHEN RespRate IS NULL THEN 1 ELSE 0 END AS RespRate_missing,
COALESCE(SpO2, 0.0) AS SpO2_value,
CASE WHEN SpO2 IS NULL THEN 1 ELSE 0 END AS SpO2_missing,
COALESCE(Temp_C, 0.0) AS Temp_C_value,
CASE WHEN Temp_C IS NULL THEN 1 ELSE 0 END AS Temp_C_missing,
COALESCE(NIBP_Mean, 0.0) AS NIBP_Mean_value,
CASE WHEN NIBP_Mean IS NULL THEN 1 ELSE 0 END AS NIBP_Mean_missing,
COALESCE(NIBP_Systolic, 0.0) AS NIBP_Systolic_value,
CASE WHEN NIBP_Systolic IS NULL THEN 1 ELSE 0 END AS NIBP_Systolic_missing,
COALESCE(NIBP_Diastolic, 0.0) AS NIBP_Diastolic_value,
CASE WHEN NIBP_Diastolic IS NULL THEN 1 ELSE 0 END AS NIBP_Diastolic_missing,
COALESCE(ABP_Mean, 0.0) AS ABP_Mean_value,
CASE WHEN ABP_Mean IS NULL THEN 1 ELSE 0 END AS 

In [16]:
peek(ddb, "CHARTEVENTS_ZEROED WHERE HeartRate_missing = 1")

┌──────────┬────────────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬────────────┬──────────────┬──────────────┬────────────────┬─────────────────┬───────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬────────────────┬──────────────────┬────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬───────────┬─────────────┬─────────────┬───────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────┬──────────────┬────────────────┬────────────────────────────┬──────────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────────┬────────────────┬──────────────────┬──────────────────────────┬──────────

In [17]:
ddb.query("SELECT COUNT(*) FROM CHARTEVENTS_PIVOTED").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1653188 │
└──────────────┘



In [18]:
# Time blocks where there is a value
# query = f"""
# SELECT COUNT(*) AS full_blocks
# FROM CHARTEVENTS_PIVOTED
# WHERE HeartRate IS NOT NULL
#     AND RespRate IS NOT NULL
#     AND O2Sat IS NOT NULL
#     AND SysBP_NonInvasive IS NOT NULL
#     AND DiasBP_NonInvasive IS NOT NULL
#     AND MeanBP_NonInvasive IS NOT NULL;
# """
# ddb.query(query).show()

Same for OUTPUTEVENTS but now SUM

In [19]:
from mimic_pipeline.feature_selection import downselect_outputevents
downselect_outputevents(ddb, from_table="CLEAN_OUTPUTEVENTS", output_table="OUTPUTEVENTS_PIVOTED")

Created OUTPUTEVENTS_PIVOTED.
   └─ wrote outputevents_pivoted.parquet


In [20]:
peek(ddb, "OUTPUTEVENTS_PIVOTED")

┌──────────┬────────────────┬───────────────────┬──────────────────┬────────────────────────┬──────────────────────────┬────────────────────────┬────────────────┬───────────┬───────────┬───────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────┬─────────────────┐
│ stay_id  │ time_bin_index │ UrineOutput_Foley │ UrineOutput_Void │ UrineOutput_CondomCath │ UrineOutput_StraightCath │ UrineOutput_Suprapubic │ UrineOutput_OR │ NG_Output │ OG_Output │ Ostomy_Output │ Stool_Output │ FecalBag_Output │ TFResidual_Output │ ChestTube1_Output │ JP1_DrainOutput │
│  int64   │     int32      │      double       │      double      │         double         │          double          │         double         │     double     │  double   │  double   │    double     │    double    │     double      │      double       │      double       │     double      │
├──────────┼────────────────┼───────────────────┼──────────────────┼────────────────────────┼─────────────────────────

In [21]:
zero_fill(ddb,
          select_body=build_zero_fill_select_body(outputevents_sum_features.values()),
          from_table="OUTPUTEVENTS_PIVOTED",
          output_table="OUTPUTEVENTS_ZEROED",
          print_query=True)

Query:

    CREATE OR REPLACE TABLE OUTPUTEVENTS_ZEROED AS
    SELECT
        stay_id,
        time_bin_index,
        COALESCE(UrineOutput_Foley, 0.0) AS UrineOutput_Foley_value,
CASE WHEN UrineOutput_Foley IS NULL THEN 1 ELSE 0 END AS UrineOutput_Foley_missing,
COALESCE(UrineOutput_Void, 0.0) AS UrineOutput_Void_value,
CASE WHEN UrineOutput_Void IS NULL THEN 1 ELSE 0 END AS UrineOutput_Void_missing,
COALESCE(UrineOutput_CondomCath, 0.0) AS UrineOutput_CondomCath_value,
CASE WHEN UrineOutput_CondomCath IS NULL THEN 1 ELSE 0 END AS UrineOutput_CondomCath_missing,
COALESCE(UrineOutput_StraightCath, 0.0) AS UrineOutput_StraightCath_value,
CASE WHEN UrineOutput_StraightCath IS NULL THEN 1 ELSE 0 END AS UrineOutput_StraightCath_missing,
COALESCE(UrineOutput_Suprapubic, 0.0) AS UrineOutput_Suprapubic_value,
CASE WHEN UrineOutput_Suprapubic IS NULL THEN 1 ELSE 0 END AS UrineOutput_Suprapubic_missing,
COALESCE(UrineOutput_OR, 0.0) AS UrineOutput_OR_value,
CASE WHEN UrineOutput_OR IS NULL THEN

In [22]:
peek(ddb, "OUTPUTEVENTS_ZEROED WHERE UrineOutput_Foley_missing = 1")

┌──────────┬────────────────┬─────────────────────────┬───────────────────────────┬────────────────────────┬──────────────────────────┬──────────────────────────────┬────────────────────────────────┬────────────────────────────────┬──────────────────────────────────┬──────────────────────────────┬────────────────────────────────┬──────────────────────┬────────────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────────┬─────────────────────┬───────────────────────┬────────────────────┬──────────────────────┬───────────────────────┬─────────────────────────┬─────────────────────────┬───────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┐
│ stay_id  │ time_bin_index │ UrineOutput_Foley_value │ UrineOutput_Foley_missing │ UrineOutput_Void_value │ UrineOutput_Void_missing │ UrineOutput_CondomCath_value │ UrineOutput_CondomCath_missing │ UrineOutput_StraightCath_value │ UrineOutput_S

Same for INPUTEVENTS but now SUM

In [23]:
from mimic_pipeline.feature_selection import downselect_inputevents
downselect_inputevents(ddb, from_table="CLEAN_INPUTEVENTS", output_table="INPUTEVENTS_PIVOTED")

Created INPUTEVENTS_PIVOTED.
   └─ wrote inputevents_pivoted.parquet


In [24]:
peek(ddb, "INPUTEVENTS_PIVOTED")

┌──────────┬────────────────┬────────────────┬──────────────────┬────────────────────────┬────────────────────┬───────────┬──────────┬─────────────────┬───────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────┬────────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────┬────────────────────────┬───────────────────┐
│ stay_id  │ time_bin_index │ NaCl_0_9_Bolus │ Dextrose_5_Fluid │ Solution_Generic_Fluid │      LR_Fluid      │ PO_Intake │ GT_Flush │ Piggyback_Fluid │   Propofol_Dose   │ Norepinephrine_Dose │ Phenylephrine_Dose │   Fentanyl_Dose    │ Midazolam_Dose │ Hydromorphone_Dose │ Furosemide_Dose │ InsulinRegular_Dose │ HeparinProphylaxis_Dose │ Vancomycin_Dose │ PotassiumChloride_Dose │   KCL_Bolus_mL    │
│  int64   │     int32      │     double     │      double      │         double         │       double       │  double   │  double  │     double      │      double       │       double       

In [25]:
zero_fill(ddb,
          select_body=build_zero_fill_select_body(inputevents_sum_features.values()),
          from_table="INPUTEVENTS_PIVOTED",
          output_table="INPUTEVENTS_ZEROED",
          print_query=True)

Query:

    CREATE OR REPLACE TABLE INPUTEVENTS_ZEROED AS
    SELECT
        stay_id,
        time_bin_index,
        COALESCE(NaCl_0_9_Bolus, 0.0) AS NaCl_0_9_Bolus_value,
CASE WHEN NaCl_0_9_Bolus IS NULL THEN 1 ELSE 0 END AS NaCl_0_9_Bolus_missing,
COALESCE(Dextrose_5_Fluid, 0.0) AS Dextrose_5_Fluid_value,
CASE WHEN Dextrose_5_Fluid IS NULL THEN 1 ELSE 0 END AS Dextrose_5_Fluid_missing,
COALESCE(Solution_Generic_Fluid, 0.0) AS Solution_Generic_Fluid_value,
CASE WHEN Solution_Generic_Fluid IS NULL THEN 1 ELSE 0 END AS Solution_Generic_Fluid_missing,
COALESCE(LR_Fluid, 0.0) AS LR_Fluid_value,
CASE WHEN LR_Fluid IS NULL THEN 1 ELSE 0 END AS LR_Fluid_missing,
COALESCE(PO_Intake, 0.0) AS PO_Intake_value,
CASE WHEN PO_Intake IS NULL THEN 1 ELSE 0 END AS PO_Intake_missing,
COALESCE(GT_Flush, 0.0) AS GT_Flush_value,
CASE WHEN GT_Flush IS NULL THEN 1 ELSE 0 END AS GT_Flush_missing,
COALESCE(Piggyback_Fluid, 0.0) AS Piggyback_Fluid_value,
CASE WHEN Piggyback_Fluid IS NULL THEN 1 ELSE 0 END AS 

In [26]:
peek(ddb, "INPUTEVENTS_ZEROED WHERE NaCl_0_9_Bolus_missing = 1")

┌──────────┬────────────────┬──────────────────────┬────────────────────────┬────────────────────────┬──────────────────────────┬──────────────────────────────┬────────────────────────────────┬────────────────────┬──────────────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬───────────────────────┬─────────────────────────┬─────────────────────┬───────────────────────┬───────────────────────────┬─────────────────────────────┬──────────────────────────┬────────────────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬──────────────────────────┬────────────────────────────┬───────────────────────┬─────────────────────────┬───────────────────────────┬─────────────────────────────┬───────────────────────────────┬─────────────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────────────┬────────────────────────────────┬────────────────────┬──────────────────────┐


Combine Features

In [27]:
from mimic_pipeline.feature_selection import create_static_features
create_static_features(ddb,
                       icu="ICUSTAYS",
                       admissions="ADMISSIONS",
                       patients="PATIENTS",
                       output_table="STATIC_FEATURES")

Created STATIC_FEATURES.
   └─ wrote static_features.parquet


In [28]:
peek(ddb, "STATIC_FEATURES")

┌──────────┬───────────────┬──────────┬──────────┐
│ stay_id  │ admission_age │ gender_M │ gender_F │
│  int64   │     int64     │  int32   │  int32   │
├──────────┼───────────────┼──────────┼──────────┤
│ 30781803 │            62 │        0 │        1 │
│ 30890198 │            41 │        0 │        1 │
│ 39779152 │            62 │        1 │        0 │
│ 31927012 │            74 │        0 │        1 │
│ 30720837 │            77 │        0 │        1 │
└──────────┴───────────────┴──────────┴──────────┘



In [29]:
from mimic_pipeline.feature_selection import combine_features
combine_features(ddb,
                 bins="ALL_PATIENT_BINS",
                 static_features="STATIC_FEATURES",
                 ce="CHARTEVENTS_ZEROED",
                 oe="OUTPUTEVENTS_ZEROED",
                 ie="INPUTEVENTS_ZEROED",
                 output_table="FINAL_MEASUREMENT_TABLE",
                 print_query=True)

Query:

    CREATE OR REPLACE TABLE FINAL_MEASUREMENT_TABLE AS
    SELECT
        b.stay_id,
        b.time_bin_index,
        
        -- Static features (will be duplicated for each bin)
        s.admission_age,
        s.gender_M,
        s.gender_F,
        
        -- Dynamic CHARTEVENTS features
        ce.HeartRate_value,
ce.HeartRate_missing,
ce.RespRate_value,
ce.RespRate_missing,
ce.SpO2_value,
ce.SpO2_missing,
ce.Temp_C_value,
ce.Temp_C_missing,
ce.NIBP_Mean_value,
ce.NIBP_Mean_missing,
ce.NIBP_Systolic_value,
ce.NIBP_Systolic_missing,
ce.NIBP_Diastolic_value,
ce.NIBP_Diastolic_missing,
ce.ABP_Mean_value,
ce.ABP_Mean_missing,
ce.ABP_Systolic_value,
ce.ABP_Systolic_missing,
ce.ABP_Diastolic_value,
ce.ABP_Diastolic_missing,
ce.CVP_value,
ce.CVP_missing,
ce.EtCO2_value,
ce.EtCO2_missing,
ce.BloodTemp_CCO_value,
ce.BloodTemp_CCO_missing,
ce.FiO2_value,
ce.FiO2_missing,
ce.O2Flow_value,
ce.O2Flow_missing,
ce.TidalVolume_Observed_value,
ce.TidalVolume_Observed_missing,
ce.TidalVol

In [30]:
peek(ddb, "FINAL_MEASUREMENT_TABLE")

┌──────────┬────────────────┬───────────────┬──────────┬──────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬────────────┬──────────────┬──────────────┬────────────────┬───────────────────┬───────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬────────────────┬──────────────────┬────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬───────────┬─────────────┬─────────────┬───────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────┬──────────────┬────────────────┬────────────────────────────┬──────────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────────┬────────────────┬────────────────

In [31]:
ddb.query("SELECT UrineOutput_Foley_value FROM FINAL_MEASUREMENT_TABLE WHERE UrineOutput_Foley_value IS NULL LIMIT 20").show()

┌─────────────────────────┐
│ UrineOutput_Foley_value │
│         double          │
├─────────────────────────┤
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
│                    NULL │
├─────────────────────────┤
│         20 rows         │
└─────────────────────────┘



In [32]:
from mimic_pipeline.feature_selection import update_final_missing_values
update_final_missing_values(ddb, print_query=True)

Query:

    UPDATE FINAL_MEASUREMENT_TABLE
    SET
    HeartRate_missing = CASE WHEN HeartRate_value IS NULL THEN 1 ELSE HeartRate_missing END,
HeartRate_value = COALESCE(HeartRate_value, 0),
RespRate_missing = CASE WHEN RespRate_value IS NULL THEN 1 ELSE RespRate_missing END,
RespRate_value = COALESCE(RespRate_value, 0),
SpO2_missing = CASE WHEN SpO2_value IS NULL THEN 1 ELSE SpO2_missing END,
SpO2_value = COALESCE(SpO2_value, 0),
Temp_C_missing = CASE WHEN Temp_C_value IS NULL THEN 1 ELSE Temp_C_missing END,
Temp_C_value = COALESCE(Temp_C_value, 0),
NIBP_Mean_missing = CASE WHEN NIBP_Mean_value IS NULL THEN 1 ELSE NIBP_Mean_missing END,
NIBP_Mean_value = COALESCE(NIBP_Mean_value, 0),
NIBP_Systolic_missing = CASE WHEN NIBP_Systolic_value IS NULL THEN 1 ELSE NIBP_Systolic_missing END,
NIBP_Systolic_value = COALESCE(NIBP_Systolic_value, 0),
NIBP_Diastolic_missing = CASE WHEN NIBP_Diastolic_value IS NULL THEN 1 ELSE NIBP_Diastolic_missing END,
NIBP_Diastolic_value = COALESCE(NIBP_Diastol

In [33]:
ddb.query("""
            SELECT COUNT(*) AS rows_with_nulls
            FROM FINAL_MEASUREMENT_TABLE
            WHERE EXISTS (
                SELECT 1
                FROM FINAL_MEASUREMENT_TABLE UNPIVOT (v FOR col IN (*))
                WHERE v IS NULL
            );
          """).show()

┌─────────────────┐
│ rows_with_nulls │
│      int64      │
├─────────────────┤
│               0 │
└─────────────────┘



# Generate Labels For Prediction
Died_within_icu_stay is the event to predict

Survival_time_hours is the event time

In [34]:
peek(ddb, "FINAL_MEASUREMENT_TABLE")

┌──────────┬────────────────┬───────────────┬──────────┬──────────┬─────────────────┬───────────────────┬────────────────┬──────────────────┬────────────┬──────────────┬──────────────┬────────────────┬───────────────────┬───────────────────┬─────────────────────┬───────────────────────┬──────────────────────┬────────────────────────┬────────────────┬──────────────────┬────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬───────────┬─────────────┬─────────────┬───────────────┬─────────────────────┬───────────────────────┬────────────┬──────────────┬──────────────┬────────────────┬────────────────────────────┬──────────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────────┬────────────────┬────────────────

In [35]:
peek(ddb, "COHORT")

┌────────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────┐
│ subject_id │ hadm_id  │      admittime      │      dischtime      │ survival_time_hours │ died_within_icu_stay │
│   int64    │  int64   │      timestamp      │      timestamp      │       double        │        int32         │
├────────────┼──────────┼─────────────────────┼─────────────────────┼─────────────────────┼──────────────────────┤
│   16358985 │ 24980209 │ 2188-12-31 14:43:00 │ 2189-01-01 18:48:00 │                28.0 │                    0 │
│   10234917 │ 26495171 │ 2136-07-02 10:41:00 │ 2136-07-06 16:50:00 │               102.0 │                    0 │
│   12079093 │ 24857875 │ 2138-11-20 21:20:00 │ 2138-11-24 20:47:00 │                95.0 │                    0 │
│   10534316 │ 29068641 │ 2170-08-17 18:55:00 │ 2170-08-29 21:50:00 │               291.0 │                    0 │
│   11900721 │ 20523250 │ 2188-02-28 18:09:00 │ 2188-03-07 15:54:00 │           

In [36]:
from mimic_pipeline.feature_selection import create_labels
create_labels(ddb, icu="ICUSTAYS", cohort="COHORT", output_table="LABELS")
ddb.query("SELECT * FROM LABELS WHERE event_flag = 1 LIMIT 10").show()

Created LABELS.
   └─ wrote labels.parquet
┌──────────┬────────────┬─────────────────────┬────────────────┐
│ stay_id  │ event_flag │ survival_time_hours │ event_time_bin │
│  int64   │   int32    │       double        │     int32      │
├──────────┼────────────┼─────────────────────┼────────────────┤
│ 32642450 │          1 │               147.0 │             36 │
│ 37749481 │          1 │                10.0 │              2 │
│ 31579295 │          1 │                43.0 │             10 │
│ 37689894 │          1 │               119.0 │             29 │
│ 35755284 │          1 │                15.0 │              3 │
│ 33200237 │          1 │               123.0 │             30 │
│ 31167596 │          1 │                49.0 │             12 │
│ 32994215 │          1 │               646.0 │            161 │
│ 39407507 │          1 │               160.0 │             40 │
│ 38272853 │          1 │               278.0 │             69 │
├──────────┴────────────┴─────────────────────┴

Account for 33-day window as per the paper.

Updated query is treated as censored after 33 days (198 bins).

Deaths after 33 days get masked (0 value assigned for event_flag)

event_time_bin: The time (in bins) of the event or censoring, capped at 197.

event_flag: 1 only if the patient died within the 33-day window. It will be 0 for all other cases (discharged alive, or died after 33 days).

In [37]:
from mimic_pipeline.feature_selection import create_windowed_labels
create_windowed_labels(ddb, icu="ICUSTAYS", cohort="COHORT", output_table="LABELS_33DAY")
ddb.query("SELECT * FROM LABELS_33DAY WHERE event_flag = 1 ORDER BY survival_time_hours DESC LIMIT 10").show()

Created LABELS_33DAY.
   └─ wrote labels_33day.parquet
┌──────────┬─────────────────────┬─────────────────────────┬─────────────────────┬────────────────┬────────────┐
│ stay_id  │ survival_time_hours │ original_event_time_bin │ original_event_flag │ event_time_bin │ event_flag │
│  int64   │       double        │          int32          │        int32        │     int32      │   int32    │
├──────────┼─────────────────────┼─────────────────────────┼─────────────────────┼────────────────┼────────────┤
│ 35412416 │               791.0 │                     197 │                   1 │            197 │          1 │
│ 31501334 │               791.0 │                     197 │                   1 │            197 │          1 │
│ 38958059 │               790.0 │                     197 │                   1 │            197 │          1 │
│ 31588440 │               790.0 │                     197 │                   1 │            197 │          1 │
│ 35047808 │               790.0 │       

As far as I am aware, processing should end here.

In [38]:
tables = ddb.execute("PRAGMA show_tables;").fetchall()
for t in tables:
    print(t[0])

ADMISSIONS
ALL_PATIENT_BINS
BAD_ADMISSIONS
BASE
CHARTEVENTS
CHARTEVENTS_PIVOTED
CHARTEVENTS_ZEROED
CLEAN_CHARTEVENTS
CLEAN_INPUTEVENTS
CLEAN_OUTPUTEVENTS
COHORT
DD_NORM
DIAGNOSES_ICD
D_ICD_DIAGNOSES
D_ITEMS
EXCLUDE_TITLES
FINAL_MEASUREMENT_TABLE
FINAL_TIMESERIES
ICUSTAYS
ICUSTAYS_COHORT
ICUSTAYS_COHORT_FIRST
ICUSTAYS_COHORT_HAS_CE_24H
ICUSTAYS_COHORT_HAS_IE_24H
ICUSTAYS_COHORT_HAS_OE_24H
ICUSTAYS_COHORT_REDUCED_24H
INPUTEVENTS
INPUTEVENTS_PIVOTED
INPUTEVENTS_ZEROED
LABELS
LABELS_33DAY
OUTPUTEVENTS
OUTPUTEVENTS_PIVOTED
OUTPUTEVENTS_ZEROED
PATIENTS
STATIC_FEATURES
